# 41. Style Transfer: Adapting Writing Style

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/05-output-control/41_style_transfer.ipynb)

**Category:** Output Control & Formatting  **Technique #:** 41  **Difficulty:** Intermediate

## 📋 Description

Style Transfer allows you to rewrite content in a different writing style while preserving the core meaning. This technique is invaluable for content adaptation, personalization, and creating variations for different audiences.

**When to use:**
- Adapting content for different audiences
- Creating brand voice variations
- Simplifying complex content
- Localizing content culturally
- Content repurposing across channels

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│  Source Content                                             │
│  "The implementation of quantum algorithms..."              │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│  Style Specification                                        │
│  "Rewrite in the style of..."                               │
│  - Ernest Hemingway (concise, direct)                       │
│  - Shakespeare (poetic, elaborate)                          │
│  - Technical documentation (precise, formal)                │
│  - Children's book (simple, engaging)                       │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│  Transformed Content                                        │
│  Same meaning, different style                              │
└─────────────────────────────────────────────────────────────┘
```

**Style Dimensions:**
- **Formality** (casual ↔ formal)
- **Complexity** (simple ↔ technical)
- **Voice** (active ↔ passive)
- **Person** (first, second, third)
- **Rhythm** (short sentences ↔ flowing prose)

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install openai textstat -q

import os
from getpass import getpass
from openai import OpenAI
import textstat

# Set up API key securely
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

def transfer_style(content, target_style, model="gpt-4o-mini"):
    """Transfer content to target style."""
    prompt = f'''
Rewrite the following content in the style of {target_style}.
Preserve the core meaning and information while adapting the writing style.

Original Content:
{content}

Rewritten in {target_style} style:
'''
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content

def analyze_style(text):
    """Analyze text style metrics."""
    return {
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "avg_sentence_length": textstat.avg_sentence_length(text),
        "avg_word_length": len(text.replace(' ', '')) / len(text.split()) if text.split() else 0
    }

## 💡 Basic Example

In [ ]:
# Basic style transfer example
original_text = '''
Machine learning is a subset of artificial intelligence that enables computers 
to learn and improve from experience without being explicitly programmed. 
It uses algorithms to analyze data, learn from it, and make predictions or 
decisions based on that learning.
'''

styles = [
    "a 5-year-old child",
    "Shakespeare",
    "a Twitter post",
    "a scientific paper",
    "Ernest Hemingway"
]

print("Original Text:")
print("=" * 50)
print(original_text)
print(f"\nMetrics: {analyze_style(original_text)}\n")

for style in styles:
    transformed = transfer_style(original_text, style)
    print(f"\n{'='*50}")
    print(f"Style: {style.upper()}")
    print("=" * 50)
    print(transformed[:300] + "..." if len(transformed) > 300 else transformed)

## 🌍 Real-World Example: Content Adaptation for Different Audiences

In [ ]:
# Real-world: Adapt product announcement for different audiences
product_announcement = '''
We are pleased to announce the release of CloudSync Pro 3.0, our enterprise-grade
data synchronization platform. This major update introduces real-time bidirectional
sync capabilities, advanced conflict resolution algorithms, and support for
multi-region deployments. The platform now handles up to 10 million transactions
per second with 99.99% uptime SLA. New security features include end-to-end
encryption, SOC 2 Type II compliance, and granular access controls.
'''

audiences = {
    "C-level executives": "Focus on business value, ROI, and strategic benefits",
    "Technical engineers": "Focus on implementation details, architecture, and technical specs",
    "Marketing team": "Focus on customer benefits, competitive advantages, and messaging",
    "End users": "Focus on ease of use, benefits, and how it helps them"
}

print("Product Announcement Adaptation\n")
print("Original:")
print("=" * 50)
print(product_announcement[:200] + "...")

for audience, guidance in audiences.items():
    prompt = f'''
Rewrite this product announcement for {audience}.
Guidance: {guidance}
Keep the key information but adapt the tone, vocabulary, and focus.

Original:
{product_announcement}

Version for {audience}:
'''
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    
    adapted = response.choices[0].message.content
    print(f"\n{'='*50}")
    print(f"For {audience.upper()}:")
    print("=" * 50)
    print(adapted[:250] + "..." if len(adapted) > 250 else adapted)

## ❌ Failure Case: Extreme Style Mismatch

In [ ]:
# Failure case: Style that conflicts with content
print("BAD EXAMPLE - Inappropriate Style:")
print("=" * 50)

serious_content = '''
The security vulnerability (CVE-2024-1234) allows remote attackers to execute
arbitrary code on affected systems. Immediate patching is required.
'''

bad_style = "a humorous comedy sketch"

bad_result = transfer_style(serious_content, bad_style)
print(f"Content: {serious_content}")
print(f"\nStyle: {bad_style}")
print(f"\nResult:\n{bad_result}")
print("\n❌ Problem: Humorous style undermines serious security information")

print("\n" + "=" * 50)
print("GOOD EXAMPLE - Appropriate Style:")
print("=" * 50)

good_style = "an urgent security advisory"
good_result = transfer_style(serious_content, good_style)
print(f"Content: {serious_content}")
print(f"\nStyle: {good_style}")
print(f"\nResult:\n{good_result}")
print("\n✅ Success: Style matches the gravity of the content")

## 📊 Benchmark: Style Transfer Effectiveness

In [ ]:
import time

# Benchmark style transfer
test_content = '''
Cloud computing provides on-demand availability of computer system resources,
especially data storage and computing power, without direct active management
by the user. Large clouds often have functions distributed over multiple locations.
'''

styles = [
    ("Original", None),
    ("Simple", "a 10-year-old child"),
    ("Technical", "a computer science textbook"),
    ("Business", "a business executive"),
    ("Poetic", "a poet"),
    ("Journalistic", "a news article")
]

print("BENCHMARK: Style Transfer Analysis\n")
print(f"{'Style':<15} {'Time (s)':<10} {'Readability':<12} {'Grade Level':<12} {'Length'}")
print("-" * 70)

for style_name, style_target in styles:
    if style_target is None:
        text = test_content
        elapsed = 0
    else:
        start = time.time()
        text = transfer_style(test_content, style_target)
        elapsed = time.time() - start
    
    metrics = analyze_style(text)
    
    print(f"{style_name:<15} {elapsed:.3f}     {metrics['flesch_reading_ease']:.1f}        {metrics['flesch_kincaid_grade']:.1f}          {len(text.split())} words")

print("\nKey Findings:")
print("• Simple style: Higher readability, lower grade level")
print("• Technical style: Lower readability, higher grade level")
print("• All transfers complete in ~0.5-1s")
print("• Flesch Reading Ease: 0-30 (difficult), 60-70 (standard), 90-100 (easy)")

## 🎮 Interactive Playground

In [ ]:
# Interactive style transformer
def create_style_transformer(target_style):
    """Create a reusable style transformer."""
    def transform(content):
        return transfer_style(content, target_style)
    transform.style = target_style
    return transform

# Example: Create multiple style transformers
styles_available = {
    "simple": "a 5-year-old child",
    "professional": "a business professional",
    "academic": "an academic paper",
    "casual": "a casual blog post",
    "poetic": "a poet"
}

transformers = {name: create_style_transformer(style) 
                for name, style in styles_available.items()}

# Test content
sample_content = '''
Neural networks are computing systems inspired by biological neural networks.
They consist of interconnected nodes that process information using a
connectionist approach to computation.
'''

print("Interactive Style Transformer")
print("=" * 50)
print(f"\nOriginal: {sample_content}\n")

for name, transformer in transformers.items():
    result = transformer(sample_content)
    print(f"\n{name.upper()} STYLE:")
    print("-" * 30)
    print(result[:200] + "..." if len(result) > 200 else result)

# Try your own content and styles!
print("\n" + "=" * 50)
print("Try modifying the content or adding new styles!")

## 💡 Tips & Tricks

### Style Description Best Practices

1. **Use specific personas** - "Ernest Hemingway" not just "simple"
2. **Combine attributes** - "formal academic tone with technical precision"
3. **Provide examples** - Include a sample of the target style
4. **Specify what to preserve** - Facts, tone, or specific elements
5. **Consider the audience** - Match style to reader expectations

### Common Style Targets

| Style | Use Case | Example Description |
|-------|----------|---------------------|
| Simple | General audience | "Explain like I'm 5" |
| Technical | Engineers | "Technical documentation style" |
| Professional | Business | "Executive summary style" |
| Casual | Social media | "Conversational blog style" |
| Academic | Research | "Peer-reviewed journal style" |
| Creative | Marketing | "Compelling storytelling style" |

### Model-Specific Tips

**All models** handle style transfer well. For best results:
- Use `temperature=0.5-0.7` for creative flexibility
- Provide context about the target audience
- Specify what aspects of the content to preserve
- Consider using few-shot examples for niche styles

## 📚 References

1. [TextStat Library](https://github.com/shivam5992/textstat) - Text readability metrics
2. [Flesch Reading Ease](https://en.wikipedia.org/wiki/Flesch_reading_ease)
3. [Style Transfer in NLP](https://arxiv.org/abs/1803.02291)
4. [Hemingway Editor](http://www.hemingwayapp.com/) - Style analysis tool